# Ejercicio 3 — Algoritmo Genético para el despacho de energía

Se resuelve el ejercicio 3 del documento de **Algoritmos Genéticos**. El problema pide encontrar un despacho de energía que minimice conjuntamente los costos de generación y transporte para Cali, Bogotá, Medellín y Barranquilla. El documento da capacidades de 3, 6, 5 y 4 GW/día, demandas de 4, 3, 5 y 3 GW/día, costos de transporte por ruta y costos de generación de 680, 720, 660 y 750 $/kWh. fileciteturn1file9L385-L402

Se implementa un AG con **claves aleatorias**: cada cromosoma contiene 16 genes, uno por ruta planta→ciudad. Al ordenar las claves se obtiene una prioridad de despacho y un decodificador transforma esa prioridad en una matriz de flujos que respeta capacidades y demandas.

## 1. Datos y función objetivo

Para cada ruta se suma el costo de generación de la planta al costo de transporte:

$$c_{ij}=c^{transporte}_{ij}+c^{generación}_i$$

El objetivo es minimizar:

$$C=\sum_i\sum_j x_{ij}c_{ij}$$

El documento describe precisamente el diseño de un AG mediante representación de individuos, operadores, función de aptitud, criterio de parada y parámetros. fileciteturn1file1L51-L64

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plantas = ['Planta C', 'Planta B', 'Planta M', 'Planta B2']
ciudades = ['Cali', 'Bogotá', 'Medellín', 'Barranquilla']

capacidad = np.array([3, 6, 5, 4], dtype=float)
demanda = np.array([4, 3, 5, 3], dtype=float)

transporte = np.array([
    [1, 4, 3, 6],
    [4, 1, 4, 5],
    [3, 4, 1, 4],
    [6, 5, 4, 1]
], dtype=float)

generacion = np.array([680, 720, 660, 750], dtype=float)
costo_unitario = transporte + generacion[:, None]

print("Costo unitario total:")
display(pd.DataFrame(costo_unitario, index=plantas, columns=ciudades))

## 2. Cromosoma y decodificación

El cromosoma tiene 16 claves aleatorias, correspondientes a las 16 rutas posibles. Las rutas se ordenan de menor a mayor clave y se asigna en cada una el máximo flujo disponible.

Esta representación evita que el AG produzca despachos imposibles: el decodificador nunca supera la capacidad restante de una planta ni la demanda restante de una ciudad.

In [ ]:
N = 4
RUTAS = [(i, j) for i in range(N) for j in range(N)]

def decodificar(cromosoma):
    x = np.zeros((N, N), dtype=float)
    cap = capacidad.copy()
    dem = demanda.copy()

    for k in np.argsort(cromosoma):
        i, j = RUTAS[k]
        cantidad = min(cap[i], dem[j])
        x[i, j] += cantidad
        cap[i] -= cantidad
        dem[j] -= cantidad

    return x

def costo(x):
    return float(np.sum(x * costo_unitario))

def aptitud(cromosoma):
    return 1.0 / (1.0 + costo(decodificar(cromosoma)))

def es_factible(x, tol=1e-9):
    return (
        np.all(x >= -tol)
        and np.all(x.sum(axis=1) <= capacidad + tol)
        and np.allclose(x.sum(axis=0), demanda, atol=tol)
    )

## 3. Operadores genéticos

Se utilizan:

- **Selección por torneo**.
- **Cruce de un punto**.
- **Mutación gaussiana** de las claves.
- **Elitismo** para conservar los mejores individuos.

La elección es consistente con el material, que presenta selección, cruce y mutación como los tres operadores principales. fileciteturn1file0L68-L80

In [ ]:
def seleccionar_torneo(poblacion, apt, tam_torneo=3):
    idx = np.random.choice(len(poblacion), tam_torneo, replace=False)
    return poblacion[idx[np.argmax(apt[idx])]].copy()

def cruce_un_punto(p1, p2):
    punto = np.random.randint(1, len(p1))
    h1 = np.concatenate([p1[:punto], p2[punto:]])
    h2 = np.concatenate([p2[:punto], p1[punto:]])
    return h1, h2

def mutar(cromosoma, pm=0.08, sigma=0.12):
    hijo = cromosoma.copy()
    mascara = np.random.rand(len(hijo)) < pm
    hijo[mascara] += np.random.normal(0, sigma, mascara.sum())
    return hijo

def ejecutar_ag(tam_poblacion=80, generaciones=300, pm=0.08,
                semilla=2026, elitismo=2):

    np.random.seed(semilla)
    poblacion = np.random.rand(tam_poblacion, 16)
    mejor_crom = None
    mejor_costo = np.inf
    historial = []

    for g in range(generaciones):
        costos = np.array([costo(decodificar(c)) for c in poblacion])
        apt = 1.0 / (1.0 + costos)

        i_mejor = np.argmin(costos)
        if costos[i_mejor] < mejor_costo:
            mejor_costo = costos[i_mejor]
            mejor_crom = poblacion[i_mejor].copy()

        historial.append(mejor_costo)

        elite = np.argsort(costos)[:elitismo]
        nueva = [poblacion[i].copy() for i in elite]

        while len(nueva) < tam_poblacion:
            p1 = seleccionar_torneo(poblacion, apt)
            p2 = seleccionar_torneo(poblacion, apt)
            h1, h2 = cruce_un_punto(p1, p2)

            nueva.append(mutar(h1, pm))
            if len(nueva) < tam_poblacion:
                nueva.append(mutar(h2, pm))

        poblacion = np.array(nueva[:tam_poblacion])

    return mejor_crom, mejor_costo, np.array(historial)

## 4. Ejecución del AG

In [ ]:
mejor_crom, mejor_costo, historial = ejecutar_ag(
    tam_poblacion=80,
    generaciones=300,
    pm=0.08,
    semilla=2026,
    elitismo=2
)

mejor_plan = decodificar(mejor_crom)

resultado = pd.DataFrame(mejor_plan, index=plantas, columns=ciudades)
resultado['Generado'] = mejor_plan.sum(axis=1)

print(f"Costo encontrado por el AG: {mejor_costo:,.0f}")
display(resultado)

print("Demanda recibida por ciudad:")
display(pd.Series(mejor_plan.sum(axis=0), index=ciudades))

print("¿Despacho factible?:", es_factible(mejor_plan))

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(historial)
plt.xlabel("Generación")
plt.ylabel("Mejor costo acumulado")
plt.title("Convergencia del Algoritmo Genético")
plt.grid(alpha=0.25)
plt.show()

## 5. Comprobación independiente

El AG es estocástico. Para comprobar el resultado en este problema pequeño, se formula el mismo despacho como un problema lineal de transporte usando `scipy.optimize.linprog`. Esto sirve como **referencia de validación**, no como sustituto del AG.

In [ ]:
from scipy.optimize import linprog

c = costo_unitario.ravel()

A_ub, b_ub = [], []
for i in range(N):
    fila = np.zeros(16)
    fila[i*N:(i+1)*N] = 1
    A_ub.append(fila)
    b_ub.append(capacidad[i])

A_eq, b_eq = [], []
for j in range(N):
    fila = np.zeros(16)
    fila[j::N] = 1
    A_eq.append(fila)
    b_eq.append(demanda[j])

ref = linprog(
    c,
    A_ub=np.array(A_ub), b_ub=np.array(b_ub),
    A_eq=np.array(A_eq), b_eq=np.array(b_eq),
    bounds=(0, None),
    method='highs'
)

plan_ref = ref.x.reshape(N, N)

print(f"Costo de referencia: {ref.fun:,.0f}")
display(pd.DataFrame(plan_ref, index=plantas, columns=ciudades))
print(f"Diferencia AG - referencia: {mejor_costo - ref.fun:,.0f}")

## 6. Interpretación

La solución de referencia obtiene un costo total de **10.436** en las unidades monetarias resultantes de multiplicar los GW despachados por los costos unitarios dados en el enunciado.

Una configuración óptima de referencia es:

| Planta | Cali | Bogotá | Medellín | Barranquilla | Total generado |
|---|---:|---:|---:|---:|---:|
| Planta C | 3 | 0 | 0 | 0 | 3 |
| Planta B | 1 | 3 | 0 | 2 | 6 |
| Planta M | 0 | 0 | 5 | 0 | 5 |
| Planta B2 | 0 | 0 | 0 | 1 | 1 |

Las demandas quedan exactamente satisfechas: 4, 3, 5 y 3 GW/día. Las capacidades no se exceden.

El material señala que las ejecuciones de un AG pueden producir comportamientos diferentes por la aleatoriedad y que es conveniente observar la convergencia y repetir ejecuciones. fileciteturn1file3L141-L153

In [ ]:
# Variabilidad del AG con varias semillas
semillas = [10, 20, 30, 40, 50]
costos = []

for s in semillas:
    _, cst, _ = ejecutar_ag(
        tam_poblacion=80,
        generaciones=300,
        pm=0.08,
        semilla=s,
        elitismo=2
    )
    costos.append(cst)

display(pd.DataFrame({"Semilla": semillas, "Costo AG": costos}))
print("Mejor costo de las corridas:", min(costos))
print("Costo promedio:", np.mean(costos))